In [58]:
from mini_competition.data_functions import load_data, add_zero_sales, PROCCESED_PATH
import pandas as pd

data = load_data(PROCCESED_PATH / "train.parquet" )
data = data[data["date"] > "2016-07-01"]

In [59]:
# add zeros
data = add_zero_sales(data)

In [60]:
import numpy as np

# add lag features
def add_lag_features(df : pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)
    df["log_sales"] = np.log1p(df["unit_sales"].astype("float32"))

    g = df.groupby(["store_nbr", "item_nbr"])["log_sales"]

    for lag in [16, 21, 28, 35]:
        df[f"lag_{lag}"] = g.shift(lag)

    df["rolling_mean_7"]  = g.transform(lambda s: s.shift(16).rolling(7).mean())
    df["rolling_mean_28"] = g.transform(lambda s: s.shift(16).rolling(28).mean())

    return df.copy().drop(columns=["log_sales"])
    
# add lag features
data_with_lags = add_lag_features(data)

# drop none rows
cols = ["lag_16", "lag_21","lag_28", "lag_35","rolling_mean_7", "rolling_mean_28"]
data_with_lags = data_with_lags.dropna(subset=cols, how="any")

print(data_with_lags.info())


<class 'pandas.DataFrame'>
Index: 79767504 entries, 43 to 89139095
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   date             datetime64[ms]
 1   store_nbr        int8          
 2   item_nbr         int32         
 3   unit_sales       float32       
 4   onpromotion      object        
 5   lag_16           float32       
 6   lag_21           float32       
 7   lag_28           float32       
 8   lag_35           float32       
 9   rolling_mean_7   float64       
 10  rolling_mean_28  float64       
dtypes: datetime64[ms](1), float32(5), float64(2), int32(1), int8(1), object(1)
memory usage: 4.8+ GB
None


In [61]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import TargetEncoder, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np

def clean_features(d):
    d = d.copy()
    d["onpromotion"] = d["onpromotion"].fillna(False)
    return d

def add_date_features(d):
    d = d.copy()
    # adds weekday and month feature
    d["date"] = pd.to_datetime(d["date"])
    d["weekday_num"] = d["date"].dt.day_of_week
    d["month_num"] = d["date"].dt.month
    return d.drop(columns="date")


df = data_with_lags

train = df[(df["date"] < "2017-07-31") & (df["date"] > "2016-07-31")].sample(n=2_000_000, random_state=0)
valid = df[df["date"] >= "2017-07-31"].sample(n=400_000, random_state=0)

# Use the raw item_nbr and the raw unit_sales; the pipeline handles both
features = train.columns.drop(["unit_sales"])
X_train, y_train = train[features], train["unit_sales"]
X_valid, y_valid = valid[features], valid["unit_sales"]

# Target-encode item_nbr and pass the other columns through unchanged
preprocess = ColumnTransformer(
    [("item_te", TargetEncoder(), ["item_nbr"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")   # keeps store_nbr as 'category'

model = TransformedTargetRegressor(
    regressor=make_pipeline(
        FunctionTransformer(clean_features),
        FunctionTransformer(add_date_features),
        preprocess,
        HistGradientBoostingRegressor(
            categorical_features=["store_nbr"],
            max_iter=1000, learning_rate=0.05, max_leaf_nodes=127,
            early_stopping=True,
        ),
    ),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,   # clipping can't be undone, so skip the round-trip check
)


model.fit(X_train, y_train)
pred = model.predict(X_valid)

In [52]:
np.mean(pred)

np.float64(3.7307561813239793)

In [62]:
# scoring

from sklearn.metrics import root_mean_squared_log_error

error = root_mean_squared_log_error(pred, valid["unit_sales"])
print(f"Error for random sample set is: {error}")

Error for random sample set is: 0.5694526110305672


In [63]:
# load test set and add lag
from pathlib import Path

EXTRACTED_PATH = Path("./../data") / "raw" / "extracted"

test_data = load_data(EXTRACTED_PATH / "test.csv")

test_data["unit_sales"] = None

# combine with previous history
combined = pd.concat([df[df["date"] >= "2017-06-01"], test_data], ignore_index=True)

# add zero sales and then lag features
combined_with_zeros = add_zero_sales(combined)
combined_with_lags = add_lag_features(combined_with_zeros)

combined_with_lags[combined_with_lags["id"] > 0].tail(20)

,date,store_nbr,item_nbr,unit_sales,onpromotion,lag_16,lag_21,lag_28,lag_35,rolling_mean_7,rolling_mean_28,id
20348832,2017-08-28,54,2132957,0,False,0.0,0.0,0.0,0.0,0.0,0.0,128235540.0
20348833,2017-08-29,54,2132957,0,False,0.0,0.0,0.0,0.0,0.0,0.0,128446194.0
20348834,2017-08-30,54,2132957,0,False,0.0,0.0,0.0,0.0,0.0,0.0,128656848.0
20348835,2017-08-31,54,2132957,0,False,0.0,0.0,0.0,0.0,0.0,0.0,128867502.0
20348912,2017-08-16,54,2134244,0,False,0.0,0.0,0.0,0.0,0.0,0.0,125707693.0
20348913,2017-08-17,54,2134244,0,False,0.0,0.0,0.0,0.0,0.0,0.0,125918347.0
20348914,2017-08-18,54,2134244,0,False,0.0,0.0,0.0,0.0,0.0,0.0,126129001.0
20348915,2017-08-19,54,2134244,0,False,0.0,0.0,0.0,0.0,0.0,0.0,126339655.0
20348916,2017-08-20,54,2134244,0,False,0.0,0.0,0.0,0.0,0.0,0.0,126550309.0
20348917,2017-08-21,54,2134244,0,False,0.0,0.0,0.0,0.0,0.0,0.0,126760963.0


In [ ]:
# create csv
import os
from pathlib import Path

EXTRACTED_FOLDER = Path("./../data/raw") / "extracted"
RESULT_FOLDER = Path("./../data/results")

test_set = combined_with_lags[combined_with_lags["id"] > 0]

# make predictions
features = test_set.columns.drop(["unit_sales", "id"])
test_pred = model.predict(test_set[features])

test_pred_positive = np.clip(test_pred, 0, None)

# write to csv
submission_data = pd.DataFrame({
    "id": test_set["id"].to_numpy().astype("int64"),
    "unit_sales": np.asarray(test_pred_positive, dtype=float).ravel(),
})

os.makedirs(RESULT_FOLDER, exist_ok=True)
submission_data.to_csv(RESULT_FOLDER / "lag_features_result_v1.csv", index=False)

In [57]:
data_with_lags.head(20)

,date,store_nbr,item_nbr,unit_sales,onpromotion,log_sales,lag_16,lag_21,lag_28,lag_35,rolling_mean_7,rolling_mean_28
43,2016-08-14,1,96995,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44,2016-08-15,1,96995,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
45,2016-08-16,1,96995,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
46,2016-08-17,1,96995,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
47,2016-08-18,1,96995,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
48,2016-08-19,1,96995,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49,2016-08-20,1,96995,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50,2016-08-21,1,96995,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
51,2016-08-22,1,96995,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
52,2016-08-23,1,96995,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
